# AMPL tutorial example 7
by Xingpeng Li

Converted to Python/Pyomo by Haoxiang Wan

**Lab Website:** [rpglab.github.io](https://rpglab.github.io)

In [3]:
from pyomo.environ import *

# ─────────────────────────────────────────────────────────────────
# Base path for all data files
# ─────────────────────────────────────────────────────────────────
BASE_PATH = '/Users/csab/Desktop/ECE6379_Python_Code/00_AMPL_Tutorial_Codes/'

# ─────────────────────────────────────────────────────────────────
# Helper function: parse AMPL-style .txt data files
# ─────────────────────────────────────────────────────────────────
def parse_ampl_data(filepath):

    with open(filepath, 'r') as f:
        content = f.read()

    # Remove comments and normalise whitespace
    lines = [l.strip() for l in content.splitlines()]
    lines = [l for l in lines if l and not l.startswith('#')]
    text  = ' '.join(lines)

    sets   = {}
    params = {}

    import re
    # Split on top-level param / set declarations
    # Each block ends with ';'
    blocks = re.split(r'(?<=;)', text)

    for block in blocks:
        block = block.strip().rstrip(';').strip()
        if not block:
            continue

        # ── param: SET_NAME: param_name := val1 val2 … ──────────
        m = re.match(r'param\s*:\s*(\w+)\s*:\s*(\w+)\s*:=\s*(.*)', block, re.S)
        if m:
            set_name, param_name, data_str = m.group(1), m.group(2), m.group(3)
            tokens = data_str.split()
            idx_list, param_dict = [], {}
            it = iter(tokens)
            for idx in it:
                val = next(it)
                key = int(idx)
                idx_list.append(key)
                param_dict[key] = float(val)
            sets[set_name]    = idx_list
            params[param_name] = param_dict
            continue

        # ── param PARAM_NAME :  COL1 COL2 … :=  row data … ─────
        m = re.match(r'param\s+(\w+)\s*:(.*?):=\s*(.*)', block, re.S)
        if m:
            param_name = m.group(1)
            col_header = m.group(2).split()
            col_indices = [int(c) for c in col_header]
            data_str   = m.group(3)
            tokens     = data_str.split()
            param_dict = {}
            n_cols     = len(col_indices)
            it         = iter(tokens)
            try:
                while True:
                    row_idx = int(next(it))
                    for col_idx in col_indices:
                        val = float(next(it))
                        param_dict[(row_idx, col_idx)] = val
            except StopIteration:
                pass
            params[param_name] = param_dict
            continue

    return {'sets': sets, 'params': params}

In [5]:
# ─────────────────────────────────────────────────────────────────
# Choose data file — uncomment one option
# ─────────────────────────────────────────────────────────────────

# Option 1a: Example 5 data (all-in-one file)
data_file = BASE_PATH + 'example_7_w_Ex5_Data_all_in_one.txt'

# Option 1b: Example 6 data (all-in-one file)
# data_file = BASE_PATH + 'example_7_w_Ex6_Data_all_in_one.txt'

# ─────────────────────────────────────────────────────────────────
# (Advanced) Option 2: load from separate files
# ─────────────────────────────────────────────────────────────────
# from functools import reduce
#
# def merge_data(*filepaths):
#     merged = {'sets': {}, 'params': {}}
#     for fp in filepaths:
#         d = parse_ampl_data(fp)
#         merged['sets'].update(d['sets'])
#         merged['params'].update(d['params'])
#     return merged
#
# parsed = merge_data(
#     BASE_PATH + 'example_7_w_Ex5_Data_a.txt',
#     BASE_PATH + 'example_7_w_Ex5_Data_b.txt',
#     BASE_PATH + 'example_7_w_Ex5_Data_c.txt',
# )

# ─────────────────────────────────────────────────────────────────
# Parse selected data file
# ─────────────────────────────────────────────────────────────────
parsed = parse_ampl_data(data_file)

I_data = parsed['sets']['I']
J_data = parsed['sets']['J']
a_data = parsed['params']['a']
b_data = parsed['params']['b']
c_data = parsed['params']['c']

print('I =', I_data)
print('J =', J_data)
print('a =', a_data)
print('b =', b_data)
print('c =', c_data)

I = [1, 2]
J = [1, 2]
a = {(1, 1): 3.0, (1, 2): 4.0, (2, 1): 7.0, (2, 2): 4.0}
b = {1: 24.0, 2: 28.0}
c = {1: 1.0, 2: 1.0}


In [7]:
# ─────────────────────────────────────────────────────────────────
# Build and solve Pyomo model
# ─────────────────────────────────────────────────────────────────

# Create model
model = ConcreteModel()

# Declare sets
model.I = Set(initialize=I_data)
model.J = Set(initialize=J_data)

# Declare Parameters
model.a = Param(model.I, model.J, initialize=a_data)
model.b = Param(model.I, initialize=b_data)
model.c = Param(model.J, initialize=c_data)

# Declare Variables
model.x = Var(model.J, within=NonNegativeReals)

# Define objective function
def obj_rule(model):
    return sum(model.c[j] * model.x[j] for j in model.J)
model.obj = Objective(rule=obj_rule, sense=maximize)

# Define constraints
def constName_rule(model, i):
    return sum(model.a[i, j] * model.x[j] for j in model.J) <= model.b[i]
model.constName = Constraint(model.I, rule=constName_rule)

# Solver setting
solver = SolverFactory('gurobi')
solver.options['mipgap'] = 0.0
solver.options['timelimit'] = 90

# Solve
results = solver.solve(model, tee=True)

# Display results
print('x:')
for j in model.J:
    print(f'  x[{j}] = {model.x[j].value}')

Set parameter Username
Set parameter LicenseID to value 2708162
Academic license - for non-commercial use only - expires 2026-09-12
Read LP format model from file /var/folders/zx/xyzw1dz90lv0dgc7dt781qf80000gn/T/tmpn53uc_6z.pyomo.lp
Reading time = 0.00 seconds
x1: 2 rows, 2 columns, 4 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.3.0 25D125)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  90
MIPGap  0

Optimize a model with 2 rows, 2 columns and 4 nonzeros
Model fingerprint: 0x2275f4a2
Coefficient statistics:
  Matrix range     [3e+00, 7e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+01, 3e+01]
Presolve time: 0.00s
Presolved: 2 rows, 2 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.0000000e+30   3.125